In [54]:
# ── Librerías ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import janitor
import sqlalchemy as sa
import os
from pathlib import Path
from dotenv import load_dotenv


%matplotlib inline
# ── Librerías ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import janitor
import sqlalchemy as sa
import os
from pathlib import Path
from dotenv import load_dotenv

%matplotlib inline

# ── Opciones de visualización ─────────────────────────────────────────
# Desactivar notación científica
pd.set_option('display.float_format', '{:.3f}'.format)
np.set_printoptions(suppress=True)

# Ver todas las columnas al imprimir un DataFrame
pd.set_option('display.max_columns', None)

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ── Variables de entorno ──────────────────────────────────────────────
load_dotenv(RAIZ / '.env')

print('✅ Entorno listo')
print(f'   Raíz del proyecto: {RAIZ}')

✅ Entorno listo
   Raíz del proyecto: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Mis Casos(Importante)\PROYECTO 5 - FORECASTING RETAIL\02_FORECASTING


## IMPORTAR LOS DATOS

In [55]:
df = pd.read_pickle(ENTRENAMIENTO / '03_train_tablon_eda.pkl')
df

,store_id,item_id,dia_ind,year,month,wday,weekday,event_name_1,event_type_1,wm_yr_wk,ventas,sell_price
date,,,,,,,,,,,,
2013-01-01,CA_3,FOODS_3_090,d_704,2013,1,4,Tuesday,NewYear,National,11249,0,1.250
2013-01-01,CA_3,FOODS_3_120,d_704,2013,1,4,Tuesday,NewYear,National,11249,0,4.980
2013-01-01,CA_3,FOODS_3_202,d_704,2013,1,4,Tuesday,NewYear,National,11249,20,4.280
2013-01-01,CA_3,FOODS_3_252,d_704,2013,1,4,Tuesday,NewYear,National,11249,34,1.480
2013-01-01,CA_3,FOODS_3_329,d_704,2013,1,4,Tuesday,NewYear,National,11249,39,2.980
...,...,...,...,...,...,...,...,...,...,...,...,...
2015-11-30,CA_4,FOODS_3_329,d_1767,2015,11,3,Monday,Sin_evento,Sin_evento,11544,5,2.980
2015-11-30,CA_4,FOODS_3_555,d_1767,2015,11,3,Monday,Sin_evento,Sin_evento,11544,4,1.680
2015-11-30,CA_4,FOODS_3_586,d_1767,2015,11,3,Monday,Sin_evento,Sin_evento,11544,9,1.680


In [56]:
df.columns

Index(['store_id', 'item_id', 'dia_ind', 'year', 'month', 'wday', 'weekday',
       'event_name_1', 'event_type_1', 'wm_yr_wk', 'ventas', 'sell_price'],
      dtype='str')

## NUEVAS VARIABLES

- Ya vienen creadas con componentes de la fecha y variables de calendario
- Se vienen lags y ventanas moviles

### Variable demanda intermitente

In [57]:
#  El proposito de esta funcion es crear columnas que ayudaran al modelo a determinar si las 0 ventas que sucedieron se
#  debieron a rotura de stock o a falta de demanda por ser demanda intermitente,en este caso se usan las columnas
#  df['rotura_stock_3'], 7 y 15 dias. Funciona de la siguiente manera :

# -Recibe como parametros la columna de ventas y el numero de ventanas anteriores
# -Mascara booleana, si no hubieron ventas (0) coloca dentro de "a" el valor int de 1 o 0 si hubieron
# -Dentro de "b", suma las veces dentro de la ventana n (ej : 3) que sucedieron 0 ventas
# -Mascara booleana, Returnea "1" o "0" segun se cumple sin paron 3,7 o 15 dias dinamicamente
# Fuera de la funcion en este caso tiene 3 niveles de sospecha Posible3, MuyPosible7 y Seguro15,segun los dias sin ventas(0)

def rotura_stock(ventas, n):
    a = (ventas==0).astype(int)
    b = a.rolling(n).sum()
    return(b==n).astype(int)

#Aqui simplemente se aplica dinamicamente a nivel de producto-tienda (PxT), los resultados de la funcion
#Se aplica un for para hacerlo dinamicamente aunque haya un pequeño sacrificio al no hacerlo vectorizado

df = df.sort_values(['store_id', 'item_id', 'date'])
nivelPxT = df.groupby(['store_id', 'item_id'])['ventas']

for n in (3, 7, 15):
    df[f'rotura_stock_{n}'] = nivelPxT.transform(rotura_stock, n=n)

# Salvedades: es un indicio (proxy), no una certeza (0 ventas != falta de stock). Cuenta filas seguidas,
# no dias de calendario. El primer dia de cada serie dara 0 por falta de historico.

In [59]:
df.loc[df['rotura_stock_15'] == 1]

,store_id,item_id,dia_ind,year,month,wday,weekday,event_name_1,event_type_1,wm_yr_wk,ventas,sell_price,rotura_stock_3,rotura_stock_7,rotura_stock_15
date,,,,,,,,,,,,,,,
2013-01-15,CA_3,FOODS_3_120,d_718,2013,1,4,Tuesday,Sin_evento,Sin_evento,11251,0,4.980,1,1,1
2013-01-16,CA_3,FOODS_3_120,d_719,2013,1,5,Wednesday,Sin_evento,Sin_evento,11251,0,4.980,1,1,1
2013-01-17,CA_3,FOODS_3_120,d_720,2013,1,6,Thursday,Sin_evento,Sin_evento,11251,0,4.980,1,1,1
2013-01-18,CA_3,FOODS_3_120,d_721,2013,1,7,Friday,Sin_evento,Sin_evento,11251,0,4.980,1,1,1
2013-01-19,CA_3,FOODS_3_120,d_722,2013,1,1,Saturday,Sin_evento,Sin_evento,11252,0,4.980,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-10-09,CA_4,FOODS_3_587,d_1350,2014,10,6,Thursday,Sin_evento,Sin_evento,11436,0,2.480,1,1,1
2014-10-10,CA_4,FOODS_3_587,d_1351,2014,10,7,Friday,Sin_evento,Sin_evento,11436,0,2.480,1,1,1
2014-10-11,CA_4,FOODS_3_587,d_1352,2014,10,1,Saturday,Sin_evento,Sin_evento,11437,0,2.480,1,1,1


### Variables de lag

- ventas : lags de 15 dias
- sell_price : lags a 7 dias
- rotura_stock : lag de un dia

In [60]:
# ── Lags ─────────────────────────────────────────────────────────────
# 'columna_original': [lags en dias] -> para añadir/quitar lags solo se toca esto
CONFIG_LAGS = {
    'ventas':          list(range(1, 16)),
    'sell_price':      list(range(1, 8)),
    'rotura_stock_3':  [1],
    'rotura_stock_7':  [1],
    'rotura_stock_15': [1],
}

def crear_lags(df, config, cols_serie=('store_id', 'item_id'), col_fecha='date'):
    """Lags dentro de cada serie producto-tienda. Devuelve un df de lags (sin unir)."""
    df = df.sort_values(list(cols_serie) + [col_fecha])
    grupos = df.groupby(list(cols_serie))
    resultado = pd.DataFrame(index=df.index)

    for columna in config:
        for lag in config[columna]:
            resultado[f'{columna}_lag{lag}'] = grupos[columna].shift(lag)

    return resultado


df_lags = crear_lags(df, CONFIG_LAGS)



In [61]:
# 2) Cómo quedó df_lags
df_lags.head(20)

,ventas_lag1,ventas_lag2,ventas_lag3,ventas_lag4,ventas_lag5,ventas_lag6,ventas_lag7,ventas_lag8,ventas_lag9,ventas_lag10,ventas_lag11,ventas_lag12,ventas_lag13,ventas_lag14,ventas_lag15,sell_price_lag1,sell_price_lag2,sell_price_lag3,sell_price_lag4,sell_price_lag5,sell_price_lag6,sell_price_lag7,rotura_stock_3_lag1,rotura_stock_7_lag1,rotura_stock_15_lag1
date,,,,,,,,,,,,,,,,,,,,,,,,,
2013-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-02,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.250,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000
2013-01-03,224.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.250,1.250,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000
2013-01-04,241.000,224.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.250,1.250,1.250,NaN,NaN,NaN,NaN,0.000,0.000,0.000
2013-01-05,232.000,241.000,224.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.250,1.250,1.250,1.250,NaN,NaN,NaN,0.000,0.000,0.000
2013-01-06,301.000,232.000,241.000,224.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.250,1.250,1.250,1.250,1.250,NaN,NaN,0.000,0.000,0.000
2013-01-07,270.000,301.000,232.000,241.000,224.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.250,1.250,1.250,1.250,1.250,1.250,NaN,0.000,0.000,0.000
2013-01-08,188.000,270.000,301.000,232.000,241.000,224.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-09,215.000,188.000,270.000,301.000,232.000,241.000,224.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000


### Variables de ventanas móviles

- Se va a crear minimo movil, media movil y maximo movil con el rango de 15 dias



In [62]:
# Estas dos funciones calculan medias moviles para estadisticos y para medias moviles ponderadas, estan guardadas
# dentro de utils.py pero aqui estan definidas
# - estadistico_movil_serie pone a nivel de serie el minimo, maximo o media movil correspondiente a esa fila
# - media movil ponderada, lo mismo pero la media ponderada de ewm
def estadistico_movil_serie(ventas, n, estadistico):
    return ventas.shift(1).rolling(n).agg(estadistico)

def media_movil_ponderada_serie(ventas, n):
    return ventas.shift(1).ewm(span=n, min_periods=n).mean()

# No es SQL, toca ordenar, se crea un objeto groupby por dos dimensiones y la metrica ventas, para usar el
# "transform" que es min/max/mean over partition by(...) para no colapsar filas y ponerles su dato correspondiente
# a cada fila/serie, [Actua sobre + de 20mil filas ordenadas a nivel Producto x Tienda, como 20mil
# trabajadores en paralelo, cada uno mirando arriba con rolling solo las n filas anteriores de SU serie
# y poniendo el dato que va ahi (mean, min o max)]
# En SQL seria: ... OVER (PARTITION BY store_id, item_id ORDER BY date ROWS BETWEEN n PRECEDING AND 1 PRECEDING)

df = df.sort_values(['store_id', 'item_id', 'date'])
nivelPxT = df.groupby(['store_id', 'item_id'])['ventas']

for n in range(2, 16):
    df['ventasmin' + str(n)] = nivelPxT.transform(estadistico_movil_serie, n=n, estadistico='min')
    df['ventasmax' + str(n)] = nivelPxT.transform(estadistico_movil_serie, n=n, estadistico='max')
    df['ventasmmp' + str(n)] = nivelPxT.transform(media_movil_ponderada_serie, n=n)

In [64]:
df

,store_id,item_id,dia_ind,year,month,wday,weekday,event_name_1,event_type_1,wm_yr_wk,ventas,sell_price,rotura_stock_3,rotura_stock_7,rotura_stock_15,ventasmin2,ventasmax2,ventasmmp2,ventasmin3,ventasmax3,ventasmmp3,ventasmin4,ventasmax4,ventasmmp4,ventasmin5,ventasmax5,ventasmmp5,ventasmin6,ventasmax6,ventasmmp6,ventasmin7,ventasmax7,ventasmmp7,ventasmin8,ventasmax8,ventasmmp8,ventasmin9,ventasmax9,ventasmmp9,ventasmin10,ventasmax10,ventasmmp10,ventasmin11,ventasmax11,ventasmmp11,ventasmin12,ventasmax12,ventasmmp12,ventasmin13,ventasmax13,ventasmmp13,ventasmin14,ventasmax14,ventasmmp14,ventasmin15,ventasmax15,ventasmmp15
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2013-01-01,CA_3,FOODS_3_090,d_704,2013,1,4,Tuesday,NewYear,National,11249,0,1.250,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-02,CA_3,FOODS_3_090,d_705,2013,1,5,Wednesday,Sin_evento,Sin_evento,11249,224,1.250,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-03,CA_3,FOODS_3_090,d_706,2013,1,6,Thursday,Sin_evento,Sin_evento,11249,241,1.250,0,0,0,0.000,224.000,168.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-04,CA_3,FOODS_3_090,d_707,2013,1,7,Friday,Sin_evento,Sin_evento,11249,232,1.250,0,0,0,224.000,241.000,218.538,0.000,241.000,201.714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-05,CA_3,FOODS_3_090,d_708,2013,1,1,Saturday,Sin_evento,Sin_evento,11250,301,1.250,0,0,0,232.000,241.000,227.625,224.000,241.000,217.867,0.000,241.000,210.129,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-11-26,CA_4,FOODS_3_714,d_1763,2015,11,6,Thursday,Thanksgiving,National,11543,22,1.580,0,0,0,6.000,12.000,7.416,3.000,12.000,8.062,3.000,15.000,8.511,3.000,15.000,8.841,3.000,15.000,9.094,3.000,15.000,9.295,3.000,15.000,9.461,3.000,15.000,9.600,3.000,15.000,9.720,3.000,17.000,9.825,3.000,17.000,9.919,3.000,17.000,10.004,3.000,17.000,10.081,3.000,17.000,10.154
2015-11-27,CA_4,FOODS_3_714,d_1764,2015,11,7,Friday,Sin_evento,Sin_evento,11543,10,1.580,0,0,0,6.000,22.000,17.139,6.000,22.000,15.031,3.000,22.000,13.907,3.000,22.000,13.227,3.000,22.000,12.781,3.000,22.000,12.472,3.000,22.000,12.247,3.000,22.000,12.080,3.000,22.000,11.953,3.000,22.000,11.854,3.000,22.000,11.777,3.000,22.000,11.717,3.000,22.000,11.671,3.000,22.000,11.634
2015-11-28,CA_4,FOODS_3_714,d_1765,2015,11,1,Saturday,Sin_evento,Sin_evento,11544,17,1.580,0,0,0,10.000,22.000,12.380,6.000,22.000,12.516,6.000,22.000,12.344,3.000,22.000,12.151,3.000,22.000,11.987,3.000,22.000,11.854,3.000,22.000,11.748,3.000,22.000,11.664,3.000,22.000,11.598,3.000,22.000,11.545,3.000,22.000,11.504,3.000,22.000,11.472,3.000,22.000,11.448,3.000,22.000,11.430


In [63]:
# Revisar que df y df_lags esten alineados fila por fila antes del concat
# df_lags no trae store_id/item_id (crear_lags solo devuelve lags), la garantia
# es el indice: la funcion ordena por serie+fecha y usa ese indice en el resultado
print(df.shape[0] == df_lags.shape[0])
print(df.index.equals(df_lags.index))


True
True


## PREPARAR LOS DATASETS

### Unir todos los dataframes generados

In [65]:
df_unido = pd.concat([df,df_lags],axis=1)
df_unido.shape

(21209, 82)

### Eliminar los nulos que han generado las nuevas variables

In [66]:
df_unido = df_unido.dropna()
df_unido.shape

(20909, 82)

In [67]:
dfo = df_unido.copy()

In [68]:
dfo

,store_id,item_id,dia_ind,year,month,wday,weekday,event_name_1,event_type_1,wm_yr_wk,ventas,sell_price,rotura_stock_3,rotura_stock_7,rotura_stock_15,ventasmin2,ventasmax2,ventasmmp2,ventasmin3,ventasmax3,ventasmmp3,ventasmin4,ventasmax4,ventasmmp4,ventasmin5,ventasmax5,ventasmmp5,ventasmin6,ventasmax6,ventasmmp6,ventasmin7,ventasmax7,ventasmmp7,ventasmin8,ventasmax8,ventasmmp8,ventasmin9,ventasmax9,ventasmmp9,ventasmin10,ventasmax10,ventasmmp10,ventasmin11,ventasmax11,ventasmmp11,ventasmin12,ventasmax12,ventasmmp12,ventasmin13,ventasmax13,ventasmmp13,ventasmin14,ventasmax14,ventasmmp14,ventasmin15,ventasmax15,ventasmmp15,ventas_lag1,ventas_lag2,ventas_lag3,ventas_lag4,ventas_lag5,ventas_lag6,ventas_lag7,ventas_lag8,ventas_lag9,ventas_lag10,ventas_lag11,ventas_lag12,ventas_lag13,ventas_lag14,ventas_lag15,sell_price_lag1,sell_price_lag2,sell_price_lag3,sell_price_lag4,sell_price_lag5,sell_price_lag6,sell_price_lag7,rotura_stock_3_lag1,rotura_stock_7_lag1,rotura_stock_15_lag1
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2013-01-16,CA_3,FOODS_3_090,d_719,2013,1,5,Wednesday,Sin_evento,Sin_evento,11251,191,1.250,0,0,0,158.000,206.000,183.305,158.000,281.000,201.341,158.000,351.000,212.071,158.000,351.000,218.467,158.000,351.000,222.361,158.000,351.000,224.766,158.000,351.000,226.259,158.000,351.000,227.178,158.000,351.000,227.728,158.000,351.000,228.038,158.000,351.000,228.190,158.000,351.000,228.236,158.000,351.000,228.213,0.000,351.000,228.142,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,232.000,241.000,224.000,0.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-17,CA_3,FOODS_3_090,d_720,2013,1,6,Thursday,Sin_evento,Sin_evento,11251,170,1.250,0,0,0,158.000,191.000,188.435,158.000,206.000,196.170,158.000,281.000,203.640,158.000,351.000,209.297,158.000,351.000,213.359,158.000,351.000,216.239,158.000,351.000,218.281,158.000,351.000,219.733,158.000,351.000,220.770,158.000,351.000,221.512,158.000,351.000,222.044,158.000,351.000,222.424,158.000,351.000,222.692,158.000,351.000,222.878,191.000,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,232.000,241.000,224.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-18,CA_3,FOODS_3_090,d_721,2013,1,7,Friday,Sin_evento,Sin_evento,11251,224,1.250,0,0,0,170.000,191.000,176.145,158.000,191.000,183.085,158.000,206.000,190.182,158.000,281.000,196.185,158.000,351.000,200.930,158.000,351.000,204.592,158.000,351.000,207.400,158.000,351.000,209.557,158.000,351.000,211.224,158.000,351.000,212.522,158.000,351.000,213.540,158.000,351.000,214.347,158.000,351.000,214.990,158.000,351.000,215.507,170.000,191.000,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,232.000,241.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-19,CA_3,FOODS_3_090,d_722,2013,1,1,Saturday,Sin_evento,Sin_evento,11252,362,1.250,0,0,0,170.000,224.000,208.048,170.000,224.000,203.543,158.000,224.000,203.711,158.000,224.000,205.463,158.000,281.000,207.537,158.000,351.000,209.471,158.000,351.000,211.129,158.000,351.000,212.499,158.000,351.000,213.611,158.000,351.000,214.509,158.000,351.000,215.233,158.000,351.000,215.818,158.000,351.000,216.290,158.000,351.000,216.674,224.000,170.000,191.000,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,232.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-20,CA_3,FOODS_3_090,d_723,2013,1,2,Sunday,Sin_evento,Sin_evento,11252,255,1.250,0,0,0,224.000,362.000,310.683,170.000,362.000,282.771,170.000,362.000,267.030,158.000,362.000,257.666,158.000,362.000,251.743,158.000,362.000,247.765,158.000,362.000,244.941,158.000,362.000,242.836,158.000,362.000,241.200,158.000,362.000,239.885,158.000,362.000,238.799,158.000,362.000,237.880,158.000,362.000,237.090,158.000,362.000,236.400,362.000,224.000,170.000,191.000,158.000,206.000,281.

### Eliminar las variables que no vamos a necesitar

In [69]:
# ── Variables fuente que se descartan como predictoras ─
#  Se eliminan por dos motivos:
#   · No disponibles al predecir: sell_price es el precio del día objetivo
#     (su señal ya va en sell_price_lag1..7).
#   · Redundantes con sus derivadas o con la fecha: rotura_stock_3/7/15
#     (conservadas en su versión lag1) y dia_ind / wm_yr_wk.
dfo = dfo.drop(columns=['dia_ind','wm_yr_wk','sell_price','rotura_stock_3','rotura_stock_7','rotura_stock_15'])

In [70]:
dfo.columns

Index(['store_id', 'item_id', 'year', 'month', 'wday', 'weekday',
       'event_name_1', 'event_type_1', 'ventas', 'ventasmin2', 'ventasmax2',
       'ventasmmp2', 'ventasmin3', 'ventasmax3', 'ventasmmp3', 'ventasmin4',
       'ventasmax4', 'ventasmmp4', 'ventasmin5', 'ventasmax5', 'ventasmmp5',
       'ventasmin6', 'ventasmax6', 'ventasmmp6', 'ventasmin7', 'ventasmax7',
       'ventasmmp7', 'ventasmin8', 'ventasmax8', 'ventasmmp8', 'ventasmin9',
       'ventasmax9', 'ventasmmp9', 'ventasmin10', 'ventasmax10', 'ventasmmp10',
       'ventasmin11', 'ventasmax11', 'ventasmmp11', 'ventasmin12',
       'ventasmax12', 'ventasmmp12', 'ventasmin13', 'ventasmax13',
       'ventasmmp13', 'ventasmin14', 'ventasmax14', 'ventasmmp14',
       'ventasmin15', 'ventasmax15', 'ventasmmp15', 'ventas_lag1',
       'ventas_lag2', 'ventas_lag3', 'ventas_lag4', 'ventas_lag5',
       'ventas_lag6', 'ventas_lag7', 'ventas_lag8', 'ventas_lag9',
       'ventas_lag10', 'ventas_lag11', 'ventas_lag12', 'ventas_lag1

### Identificar la target

In [71]:
target = dfo['ventas']

### Separar num y cat

In [72]:
cat = dfo.select_dtypes(include=['str', 'category'])
num = dfo.select_dtypes(include=['number'])

## TRANSFORMACIÓN CATEGÓRICAS

### One Hot Encoding

#### Variables a aplicar OHE

In [73]:
# ── Variables a aplicar OHE ──────────────────────────────────────────
# Todas son categóricas nominales: sin orden intrínseco ni relación monótona
# con la target (verificado en el EDA) → OHE en lugar de Ordinal Encoding.
# Excepción a valorar: 'month' tiene estacionalidad (p. ej. la temporada de
# mango en Colombia cae en trimestre 1 y 4), donde un OE la capturaría mejor.

var_ohe = ['year', 'month', 'wday','weekday','event_name_1','event_type_1']

Como se va a usar LightGBM o XGBOOST, no se reescala ni normaliza, porque no es muy sensible a que la Distribucion sea mas normal o al reescalado

#### Instanciar

In [74]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

#### Entrenar y aplicar

In [75]:
cat_ohe = ohe.fit_transform(cat[var_ohe])
cat_ohe_df = pd.DataFrame(cat_ohe, columns=ohe.get_feature_names_out(var_ohe))

## UNIFICAR DATASETS TRANSFORMADOS

In [76]:
cat_ohe_df

,year_2014,year_2015,month_10,month_11,month_12,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,wday_2,wday_3,wday_4,wday_5,wday_6,wday_7,weekday_Monday,weekday_Saturday,weekday_Sunday,weekday_Thursday,weekday_Tuesday,weekday_Wednesday,event_name_1_Christmas,event_name_1_Cinco De Mayo,event_name_1_ColumbusDay,event_name_1_Easter,event_name_1_Eid al-Fitr,event_name_1_EidAlAdha,event_name_1_Father's day,event_name_1_Halloween,event_name_1_IndependenceDay,event_name_1_LaborDay,event_name_1_LentStart,event_name_1_LentWeek2,event_name_1_MartinLutherKingDay,event_name_1_MemorialDay,event_name_1_Mother's day,event_name_1_NBAFinalsEnd,event_name_1_NBAFinalsStart,event_name_1_NewYear,event_name_1_OrthodoxChristmas,event_name_1_OrthodoxEaster,event_name_1_Pesach End,event_name_1_PresidentsDay,event_name_1_Purim End,event_name_1_Ramadan starts,event_name_1_Sin_evento,event_name_1_StPatricksDay,event_name_1_SuperBowl,event_name_1_Thanksgiving,event_name_1_ValentinesDay,event_name_1_VeteransDay,event_type_1_National,event_type_1_Religious,event_type_1_Sin_evento,event_type_1_Sporting
0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000
2,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000
3,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000
4,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20904,0.000,1.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000,0.000,0.000,0.000
20905,0.000,1.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000
20906,0.000,1.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,

In [77]:
num

,ventas,ventasmin2,ventasmax2,ventasmmp2,ventasmin3,ventasmax3,ventasmmp3,ventasmin4,ventasmax4,ventasmmp4,ventasmin5,ventasmax5,ventasmmp5,ventasmin6,ventasmax6,ventasmmp6,ventasmin7,ventasmax7,ventasmmp7,ventasmin8,ventasmax8,ventasmmp8,ventasmin9,ventasmax9,ventasmmp9,ventasmin10,ventasmax10,ventasmmp10,ventasmin11,ventasmax11,ventasmmp11,ventasmin12,ventasmax12,ventasmmp12,ventasmin13,ventasmax13,ventasmmp13,ventasmin14,ventasmax14,ventasmmp14,ventasmin15,ventasmax15,ventasmmp15,ventas_lag1,ventas_lag2,ventas_lag3,ventas_lag4,ventas_lag5,ventas_lag6,ventas_lag7,ventas_lag8,ventas_lag9,ventas_lag10,ventas_lag11,ventas_lag12,ventas_lag13,ventas_lag14,ventas_lag15,sell_price_lag1,sell_price_lag2,sell_price_lag3,sell_price_lag4,sell_price_lag5,sell_price_lag6,sell_price_lag7,rotura_stock_3_lag1,rotura_stock_7_lag1,rotura_stock_15_lag1
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2013-01-16,191,158.000,206.000,183.305,158.000,281.000,201.341,158.000,351.000,212.071,158.000,351.000,218.467,158.000,351.000,222.361,158.000,351.000,224.766,158.000,351.000,226.259,158.000,351.000,227.178,158.000,351.000,227.728,158.000,351.000,228.038,158.000,351.000,228.190,158.000,351.000,228.236,158.000,351.000,228.213,0.000,351.000,228.142,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,232.000,241.000,224.000,0.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-17,170,158.000,191.000,188.435,158.000,206.000,196.170,158.000,281.000,203.640,158.000,351.000,209.297,158.000,351.000,213.359,158.000,351.000,216.239,158.000,351.000,218.281,158.000,351.000,219.733,158.000,351.000,220.770,158.000,351.000,221.512,158.000,351.000,222.044,158.000,351.000,222.424,158.000,351.000,222.692,158.000,351.000,222.878,191.000,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,232.000,241.000,224.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-18,224,170.000,191.000,176.145,158.000,191.000,183.085,158.000,206.000,190.182,158.000,281.000,196.185,158.000,351.000,200.930,158.000,351.000,204.592,158.000,351.000,207.400,158.000,351.000,209.557,158.000,351.000,211.224,158.000,351.000,212.522,158.000,351.000,213.540,158.000,351.000,214.347,158.000,351.000,214.990,158.000,351.000,215.507,170.000,191.000,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,232.000,241.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-19,362,170.000,224.000,208.048,170.000,224.000,203.543,158.000,224.000,203.711,158.000,224.000,205.463,158.000,281.000,207.537,158.000,351.000,209.471,158.000,351.000,211.129,158.000,351.000,212.499,158.000,351.000,213.611,158.000,351.000,214.509,158.000,351.000,215.233,158.000,351.000,215.818,158.000,351.000,216.290,158.000,351.000,216.674,224.000,170.000,191.000,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,232.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
2013-01-20,255,224.000,362.000,310.683,170.000,362.000,282.771,170.000,362.000,267.030,158.000,362.000,257.666,158.000,362.000,251.743,158.000,362.000,247.765,158.000,362.000,244.941,158.000,362.000,242.836,158.000,362.000,241.200,158.000,362.000,239.885,158.000,362.000,238.799,158.000,362.000,237.880,158.000,362.000,237.090,158.000,362.000,236.400,362.000,224.000,170.000,191.000,158.000,206.000,281.000,351.000,214.000,245.000,197.000,215.000,188.000,270.000,301.000,1.250,1.250,1.250,1.250,1.250,1.250,1.250,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-11-26,22,6.000,12.000,7.416,3.000,12.000,8.062,3.000,15.000,8.511,3.000,15.000,8.841,3.000,15.000,9.094,3.000,15.000,9.295,3.000,15.000,9.461,3.000,

### Meter en una lista todos los dataframes generados

### Unir todos los dataframes

In [78]:
# ── Unir los datasets transformados ───────────────────────────────────
# cat y num salen del mismo dfo y conservan el orden de filas, así que el
# índice de cat es el que debe recuperar cat_ohe_df para poder alinearlos
cat_ohe_df.index = cat.index

# store_id / item_id se quedaron en cat (son str)
claves = cat[['store_id', 'item_id']]

# num ya incluye 'ventas' -> se saca aparte para no duplicar la target
df = pd.concat([claves, num.drop(columns=['ventas']), cat_ohe_df, target], axis=1)

# el 'date' deja de ser índice y pasa a columna
df = df.reset_index()

# assert: comprueba un supuesto y lanza AssertionError si no se cumple.
# Es un contrato rápido: si algo dejó de cuadrar, salta aquí y no más adelante
assert df.shape[0] == num.shape[0]
assert df.isnull().sum().sum() == 0
df.shape

(20909, 130)

## GUARDAR DATASET TRAS TRANSFORMACIÓN

In [46]:
df.to_pickle(ENTRENAMIENTO/'df_tablon_transformado.pkl')

In [48]:
# ── Estructura final del tablón transformado (documentación de la fase) ──
print('Shape:', df.shape)
print('Primeras columnas:', df.columns[:8].tolist())
print('Rango de fechas:', df['date'].min(), '→', df['date'].max())

bloques = ['ventas_lag', 'sell_price_lag', 'rotura_stock_', 'ventasmin', 'ventasmax', 'ventasmmp',
           'year_', 'month_', 'wday_', 'weekday_', 'event_name_1_', 'event_type_1_']
for prefijo in bloques:
    print(f'{prefijo:<15} {int(df.columns.str.startswith(prefijo).sum()):>3}')

df.info(verbose=True)

Shape: (20909, 130)
Primeras columnas: ['date', 'store_id', 'item_id', 'ventasmin2', 'ventasmax2', 'ventasmmp2', 'ventasmin3', 'ventasmax3']
Rango de fechas: 2013-01-16 → 2015-11-30
ventas_lag       15
sell_price_lag    7
rotura_stock_     3
ventasmin        14
ventasmax        14
ventasmmp        14
year_             2
month_           11
wday_             6
weekday_          6
event_name_1_    30
event_type_1_     4
<class 'pandas.DataFrame'>
RangeIndex: 20909 entries, 0 to 20908
Data columns (total 130 columns):
 #    Column                            Dtype  
---   ------                            -----  
 0    date                              str    
 1    store_id                          str    
 2    item_id                           str    
 3    ventasmin2                        float64
 4    ventasmax2                        float64
 5    ventasmmp2                        float64
 6    ventasmin3                        float64
 7    ventasmax3                        float64